# 01.1 — Análise Temporal de Retorno + Comorbidades
**Depende de:** `01_eda_gestante.py` executado (tabela `gestante_enriquecido` no DuckDB)

Três blocos independentes:
1. **Tempo ao retorno** — Kaplan-Meier–like: curva cumulativa de retorno por dias desde a alta
2. **Retorno 30d/90d sem corte por ano** — distribuição do desfecho como coorte única
3. **Comorbidades** — frequência, co-ocorrência e associação com retorno

In [1]:
from __future__ import annotations
import warnings
from datetime import datetime
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60)
pd.set_option("display.float_format", "{:.2f}".format)

ROOT     = Path("..").resolve()
DUCKDB   = ROOT / "data" / "gestante.duckdb"
OUT_DIR  = ROOT / "out" / "01.1_temporal_comorbidades"
PLOT_DIR = OUT_DIR / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ── Idioma / Language ─────────────────────────────────────────────────────────
LANG = "pt"  # "pt" | "en"  ← única linha a alterar para mudar idioma

_T = {
    "pt": {
        # Curva de retorno
        "curva_title":        "Curva Cumulativa de Retorno\n(coorte completa 2017–2023)",
        "curva_grupo_title":  "Curva de Retorno por Grupo Clínico",
        "curva_suptitle":     "Tempo ao Retorno — Independente do Ano de Atendimento",
        "xlabel_dias_alta":   "Dias desde a alta",
        "ylabel_retorno_pct": "Pacientes que retornaram (%)",
        "ylabel_retorno_acum":"Retorno acumulado (%)",
        "lbl_30d":            "30d",
        "lbl_90d":            "90d",
        # Distribuição de retorno
        "hist_title":         "Distribuição do Tempo ao Retorno\n(0–90 dias)",
        "hist_xlabel":        "Dias até o retorno",
        "hist_ylabel":        "Número de atendimentos",
        "bar_30d_lbl":        "30 dias",
        "bar_90d_lbl":        "90 dias",
        "barra_title":        "Taxa de Retorno 30d e 90d\npor Grupo Clínico (coorte única)",
        "barra_ylabel":       "Taxa de retorno (%)",
        "ano_title":          "Estabilidade da Taxa de Retorno\npor Ano",
        "ano_xlabel":         "Ano",
        "ano_ylabel":         "Taxa de retorno (%)",
        "dist_suptitle":      "Análise de Retorno — Perspectiva Temporal Pós-Alta",
        # Comorbidades
        "freq_title":         "Frequência dos Grupos Clínicos\n(CID principal)",
        "freq_xlabel":        "% dos atendimentos",
        "top_cid_title":      "Top 25 CIDs mais frequentes",
        "comorb_suptitle":    "Perfil de Comorbidades — Coorte de Gestantes de Alto Risco",
        "diag_title":         "Distribuição: Diagnósticos por Atendimento",
        "diag_xlabel":        "Número de diagnósticos registrados",
        "diag_ylabel":        "Número de atendimentos",
        "prob_title":         "Distribuição: Problemas Crônicos por Paciente",
        "prob_xlabel":        "Número de problemas crônicos (PW_PROBLEMA_PACIENTE)",
        "prob_ylabel":        "Número de pacientes",
        "carga_suptitle":     "Carga de Comorbidades por Atendimento",
        "assoc_title":        "Taxa de Retorno 30d por Grupo Clínico\ncom IC 95% Wilson",
        "assoc_ylabel":       "Taxa de retorno em 30d (%)",
        "assoc_media_lbl":    "Média geral ({pct:.1f}%)",
        "sec_title":          "Top 15 CIDs Secundários mais Frequentes\n(diagnósticos + laudo AIH)",
        "sec_xlabel":         "Prefixo CID (3 chars)",
        "sec_ylabel":         "% de atendimentos com esse CID",
        # Sazonalidade
        "saz_a_suptitle":     "Distribuição Mensal de Patologias por Ano\n(barras empilhadas = % do total de atendimentos naquele mês)",
        "saz_a_ylabel":       "% do mês",
        "saz_b_suptitle":     "Sazonalidade por Grupo Clínico — Anos Sobrepostos\n(% do total do mês | tracejado = 2023)",
        "saz_b_ylabel":       "% do mês",
        "mes_xlabel":         "Mês",
        "meses":              ["Jan","Fev","Mar","Abr","Mai","Jun","Jul","Ago","Set","Out","Nov","Dez"],
    },
    "en": {
        # Return curve
        "curva_title":        "Cumulative Return Curve\n(full cohort 2017–2023)",
        "curva_grupo_title":  "Return Curve by Clinical Group",
        "curva_suptitle":     "Time to Return — Year-Independent Analysis",
        "xlabel_dias_alta":   "Days since discharge",
        "ylabel_retorno_pct": "Patients returned (%)",
        "ylabel_retorno_acum":"Cumulative return (%)",
        "lbl_30d":            "30d",
        "lbl_90d":            "90d",
        # Return distribution
        "hist_title":         "Distribution of Time to Return\n(0–90 days)",
        "hist_xlabel":        "Days to return",
        "hist_ylabel":        "Number of attendances",
        "bar_30d_lbl":        "30 days",
        "bar_90d_lbl":        "90 days",
        "barra_title":        "30d and 90d Return Rate\nby Clinical Group (pooled cohort)",
        "barra_ylabel":       "Return rate (%)",
        "ano_title":          "Return Rate Stability\nby Year",
        "ano_xlabel":         "Year",
        "ano_ylabel":         "Return rate (%)",
        "dist_suptitle":      "Return Analysis — Post-Discharge Temporal Perspective",
        # Comorbidities
        "freq_title":         "Clinical Group Frequency\n(Primary ICD-10)",
        "freq_xlabel":        "% of attendances",
        "top_cid_title":      "Top 25 most frequent ICD-10 codes",
        "comorb_suptitle":    "Comorbidity Profile — High-Risk Obstetric Cohort",
        "diag_title":         "Distribution: Diagnoses per Attendance",
        "diag_xlabel":        "Number of recorded diagnoses",
        "diag_ylabel":        "Number of attendances",
        "prob_title":         "Distribution: Chronic Problems per Patient",
        "prob_xlabel":        "Number of chronic problems (problem list)",
        "prob_ylabel":        "Number of patients",
        "carga_suptitle":     "Comorbidity Burden per Attendance",
        "assoc_title":        "30-day Return Rate by Clinical Group\nwith 95% Wilson CI",
        "assoc_ylabel":       "30-day return rate (%)",
        "assoc_media_lbl":    "Overall mean ({pct:.1f}%)",
        "sec_title":          "Top 15 Most Frequent Secondary ICD-10 Codes\n(diagnoses + AIH report)",
        "sec_xlabel":         "ICD-10 prefix (3 chars)",
        "sec_ylabel":         "% of attendances with this code",
        # Seasonality
        "saz_a_suptitle":     "Monthly Distribution of Pathologies by Year\n(stacked bars = % of total attendances in that month)",
        "saz_a_ylabel":       "% of month",
        "saz_b_suptitle":     "Seasonality by Clinical Group — Overlapping Years\n(% of monthly total | dashed = 2023)",
        "saz_b_ylabel":       "% of month",
        "mes_xlabel":         "Month",
        "meses":              ["Jan","Feb","Mar","Apr","May","Jun","Jul","Aug","Sep","Oct","Nov","Dec"],
    },
}
T = _T[LANG]

# Paleta consistente com os outros notebooks
CORES_GRUPO = {
    "Hipertensivos":      "#e63946",
    "Diabetes Gest.":     "#f4a261",
    "Pré-natal":          "#2a9d8f",
    "Parto Prematuro":    "#457b9d",
    "Hemorragia":         "#8338ec",
    "Comorbidades":       "#fb8500",
    "Outros Obstétricos": "#adb5bd",
    "Infecção":           "#606c38",
}

# ── Mapeamento CID → grupo clínico ────────────────────────────────────────────
def mapeia_grupo_cid(cid: str | None) -> str:
    if not cid:
        return "Outros Obstétricos"
    c = str(cid).upper().strip()
    if c[:3] in ("O10","O11","O13","O14","O15","O16"): return "Hipertensivos"
    if c[:3] in ("O24","O244","O249"):                  return "Diabetes Gest."
    if c[:3] in ("O47","O60","O61","O62","O63","O471","O470"): return "Parto Prematuro"
    if c[:2] == "Z3":                                   return "Pré-natal"
    if c[:3] in ("O20","O44","O45","O46","O67","O72"): return "Hemorragia"
    if c[:3] in ("O85","O86","O87","O88","O89","O91","A","B"): return "Infecção"
    if c[:3] in ("O99","O94","O95","O96","O97","O98"): return "Comorbidades"
    return "Outros Obstétricos"

## 1. Carregar dados

In [2]:
con = duckdb.connect(str(DUCKDB), read_only=True)

df = con.execute("""
    SELECT cd_atendimento, cd_paciente,
           dt_atendimento, dt_alta,
           sn_retorno_30d, sn_retorno_90d,
           cd_cid, cids_relacionados, cids_diagnostico,
           cid_principal_laudo, cid_secundario_laudo,
           qtd_diagnosticos, qtd_problemas
    FROM gestante_enriquecido
    WHERE dt_atendimento IS NOT NULL
      AND dt_alta IS NOT NULL
""").df()

# Dias ao retorno via self-join (até 90 dias)
df_dias = con.execute("""
    SELECT a.cd_atendimento,
           DATEDIFF('day', ANY_VALUE(a.dt_alta), MIN(b.dt_atendimento)) AS dias_ate_retorno
    FROM gestante_enriquecido a
    JOIN gestante_enriquecido b
      ON b.cd_paciente = a.cd_paciente
     AND b.dt_atendimento > a.dt_alta
     AND b.dt_atendimento <= a.dt_alta + INTERVAL 90 DAY
    WHERE a.dt_alta IS NOT NULL
      AND b.dt_atendimento IS NOT NULL
    GROUP BY a.cd_atendimento
""").df()

con.close()

df = df.merge(df_dias, on="cd_atendimento", how="left")

# Extrai ano e mês diretamente do campo de data via pandas
_dt = pd.to_datetime(df["dt_atendimento"], errors="coerce")
df["ano"] = _dt.dt.year
df["mes"] = _dt.dt.month

df["y_30d"]  = (df["sn_retorno_30d"] == "S").astype(int)
df["y_90d"]  = (df["sn_retorno_90d"] == "S").astype(int)
df["grupo"]  = df["cd_cid"].apply(mapeia_grupo_cid)
df["retornou_90d"] = df["dias_ate_retorno"].notna().astype(int)

n_total = len(df)
n_com_dias = df["dias_ate_retorno"].notna().sum()
print(f"Total de atendimentos: {n_total:,}")
print(f"Com dias ao retorno calculado (≤90d): {n_com_dias:,} ({n_com_dias/n_total:.1%})")
print(f"Taxa retorno 30d: {df['y_30d'].mean():.1%}")
print(f"Taxa retorno 90d: {df['y_90d'].mean():.1%}")

## 2. Curva Cumulativa de Retorno por Dias desde a Alta

**Independente do ano de atendimento** — toda a coorte tratada como um único pool.
Eixo X: dias corridos desde a alta. Eixo Y: proporção acumulada que retornou.

In [3]:
# Curva cumulativa geral
MAX_DIA = 90
dias_vals = np.arange(0, MAX_DIA + 1)

retornos_com_dia = df.dropna(subset=["dias_ate_retorno"]).copy()
retornos_com_dia["dias_ate_retorno"] = retornos_com_dia["dias_ate_retorno"].clip(0, MAX_DIA)

# Para cada dia d: proporção do total que retornou até d
retorno_cum = np.array([
    (retornos_com_dia["dias_ate_retorno"] <= d).sum() / n_total
    for d in dias_vals
])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Plot 1: curva geral ──
ax = axes[0]
ax.plot(dias_vals, retorno_cum * 100, color="#1d3557", lw=2.5)
ax.axvline(30, color="tomato",   ls="--", lw=1.5, label=f"{T['lbl_30d']}  ({retorno_cum[30]*100:.1f}%)")
ax.axvline(90, color="darkorange", ls="--", lw=1.5, label=f"{T['lbl_90d']}  ({retorno_cum[90]*100:.1f}%)")
ax.fill_between(dias_vals, retorno_cum * 100, alpha=0.12, color="#1d3557")
ax.set_xlabel(T["xlabel_dias_alta"], fontsize=11)
ax.set_ylabel(T["ylabel_retorno_pct"], fontsize=11)
ax.set_title(T["curva_title"], fontsize=11)
ax.legend(fontsize=10); ax.grid(alpha=0.3)
ax.set_xlim(0, MAX_DIA); ax.set_ylim(0, None)

# ── Plot 2: curva por grupo clínico ──
ax2 = axes[1]
grupos_ordem = df["grupo"].value_counts().index.tolist()
for grupo in grupos_ordem:
    sub = df[df["grupo"] == grupo]
    sub_dias = sub.dropna(subset=["dias_ate_retorno"])
    n_sub = len(sub)
    if n_sub < 50:
        continue
    cum = np.array([(sub_dias["dias_ate_retorno"] <= d).sum() / n_sub for d in dias_vals])
    cor = CORES_GRUPO.get(grupo, "#888888")
    ax2.plot(dias_vals, cum * 100, lw=2, color=cor, label=f"{grupo} (n={n_sub:,})")

ax2.axvline(30, color="gray", ls="--", lw=1, alpha=0.6)
ax2.axvline(90, color="gray", ls=":",  lw=1, alpha=0.6)
ax2.set_xlabel(T["xlabel_dias_alta"], fontsize=11)
ax2.set_ylabel(T["ylabel_retorno_acum"], fontsize=11)
ax2.set_title(T["curva_grupo_title"], fontsize=11)
ax2.legend(fontsize=8, loc="upper left"); ax2.grid(alpha=0.3)
ax2.set_xlim(0, MAX_DIA); ax2.set_ylim(0, None)

plt.suptitle(T["curva_suptitle"], fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / "01_curva_retorno_cumulativo.png", dpi=150, bbox_inches="tight")
plt.close()
print("Gerado: 01_curva_retorno_cumulativo.png")

# Tabela resumo por marco temporal
marcos = [7, 14, 21, 30, 45, 60, 90]
rows_curva = []
for grupo in ["(geral)"] + grupos_ordem:
    if grupo == "(geral)":
        sub = df
    else:
        sub = df[df["grupo"] == grupo]
    sub_dias = sub.dropna(subset=["dias_ate_retorno"])
    n_sub = len(sub)
    row = {"grupo": grupo, "n": n_sub}
    for d in marcos:
        pct = (sub_dias["dias_ate_retorno"] <= d).sum() / n_sub * 100
        row[f"retorno_{d}d_pct"] = round(pct, 2)
    rows_curva.append(row)

df_curva = pd.DataFrame(rows_curva)
df_curva.to_csv(OUT_DIR / "retorno_cumulativo_por_grupo.csv", index=False)
print(df_curva.to_string(index=False))

## 3. Distribuição do Retorno 30d/90d — Coorte Única (sem corte por ano)

Em vez de dividir pelo calendário, olha o desfecho como distribuição
no nível do evento: **quanto tempo após a alta o paciente retornou?**

In [4]:
fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5))

# ── Plot 1: histograma dias ao retorno (0–90) ──
ax = axes2[0]
dados_hist = df["dias_ate_retorno"].dropna().clip(1, 90)
ax.hist(dados_hist, bins=45, color="#457b9d", edgecolor="white", lw=0.4)
ax.axvline(30, color="tomato",   ls="--", lw=1.5, label=T["bar_30d_lbl"])
ax.axvline(90, color="darkorange", ls="--", lw=1.5, label=T["bar_90d_lbl"])
ax.set_xlabel(T["hist_xlabel"], fontsize=11)
ax.set_ylabel(T["hist_ylabel"], fontsize=11)
ax.set_title(T["hist_title"], fontsize=11)
ax.legend(fontsize=10); ax.grid(alpha=0.3)

# ── Plot 2: taxa 30d e 90d por grupo clínico (sem distinção de ano) ──
ax2 = axes2[1]
taxa_grupo = (df.groupby("grupo")
               .agg(n=("y_30d","count"),
                    taxa_30d=("y_30d","mean"),
                    taxa_90d=("y_90d","mean"))
               .reset_index()
               .sort_values("taxa_30d", ascending=False))
taxa_grupo["taxa_30d_pct"] = taxa_grupo["taxa_30d"] * 100
taxa_grupo["taxa_90d_pct"] = taxa_grupo["taxa_90d"] * 100

x = np.arange(len(taxa_grupo))
w = 0.35
bars1 = ax2.bar(x - w/2, taxa_grupo["taxa_30d_pct"], w, label=T["bar_30d_lbl"],
                color="#457b9d", edgecolor="white")
bars2 = ax2.bar(x + w/2, taxa_grupo["taxa_90d_pct"], w, label=T["bar_90d_lbl"],
                color="#e9c46a", edgecolor="white")
ax2.set_xticks(x)
ax2.set_xticklabels(taxa_grupo["grupo"], rotation=35, ha="right", fontsize=8)
ax2.set_ylabel(T["barra_ylabel"], fontsize=11)
ax2.set_title(T["barra_title"], fontsize=11)
ax2.legend(fontsize=9); ax2.grid(axis="y", alpha=0.3)
for b in bars1: ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f"{b.get_height():.0f}%", ha="center", fontsize=7)
for b in bars2: ax2.text(b.get_x()+b.get_width()/2, b.get_height()+0.3, f"{b.get_height():.0f}%", ha="center", fontsize=7)

# ── Plot 3: estabilidade anual (confirma que o padrão não muda por ano) ──
ax3 = axes2[2]
taxa_ano = (df.groupby("ano")
              .agg(taxa_30d=("y_30d","mean"),
                   taxa_90d=("y_90d","mean"),
                   n=("y_30d","count"))
              .reset_index())
ax3.plot(taxa_ano["ano"], taxa_ano["taxa_30d"]*100, "o-", color="#457b9d", lw=2, label=T["bar_30d_lbl"])
ax3.plot(taxa_ano["ano"], taxa_ano["taxa_90d"]*100, "s-", color="#e9c46a", lw=2, label=T["bar_90d_lbl"])
ax3.set_xlabel(T["ano_xlabel"], fontsize=11)
ax3.set_ylabel(T["ano_ylabel"], fontsize=11)
ax3.set_title(T["ano_title"], fontsize=11)
ax3.legend(fontsize=10); ax3.grid(alpha=0.3)

plt.suptitle(T["dist_suptitle"], fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / "02_distribuicao_retorno.png", dpi=150, bbox_inches="tight")
plt.close()
print("Gerado: 02_distribuicao_retorno.png")

# Estatísticas descritivas do tempo ao retorno
desc = df["dias_ate_retorno"].describe(percentiles=[.25, .5, .75, .9])
print("\n=== Estatísticas: dias ao retorno ===")
print(desc)

taxa_grupo.to_csv(OUT_DIR / "taxa_retorno_por_grupo.csv", index=False)
taxa_ano.to_csv(OUT_DIR / "taxa_retorno_por_ano.csv", index=False)

## 4. Comorbidades — Frequência, Co-ocorrência e Associação com Retorno

Fontes de CID usadas:
- `cd_cid` — CID principal do atendimento
- `cids_diagnostico` — CIDs do DIAGNOSTICO_ATENDIME
- `cid_principal_laudo`, `cid_secundario_laudo` — laudo AIH

In [5]:
# ── 4a. Frequência dos grupos clínicos ────────────────────────────────────────
fig3, axes3 = plt.subplots(1, 2, figsize=(14, 5))

ax = axes3[0]
freq_grupo = df["grupo"].value_counts().reset_index()
freq_grupo.columns = ["grupo", "n"]
freq_grupo["pct"] = freq_grupo["n"] / n_total * 100
cores_bar = [CORES_GRUPO.get(g, "#adb5bd") for g in freq_grupo["grupo"]]
bars = ax.barh(freq_grupo["grupo"], freq_grupo["pct"], color=cores_bar, edgecolor="white")
ax.set_xlabel(T["freq_xlabel"], fontsize=11)
ax.set_title(T["freq_title"], fontsize=11)
ax.grid(axis="x", alpha=0.3)
for b in bars:
    w = b.get_width()
    ax.text(w + 0.2, b.get_y() + b.get_height()/2, f"{w:.1f}%", va="center", fontsize=9)

# ── 4b. Top 25 CIDs individuais ───────────────────────────────────────────────
ax2 = axes3[1]
top_cids = (df["cd_cid"]
            .value_counts()
            .head(25)
            .reset_index())
top_cids.columns = ["cd_cid", "n"]
top_cids["pct"] = top_cids["n"] / n_total * 100

# Descrições resumidas
desc_cid = {
    "O809": "Parto normal espontâneo",
    "O13":  "Hipertensão gestacional",
    "O998": "Doenças maternas (outras)",
    "Z349": "Vigilância pré-natal",
    "O200": "Ameaça de aborto",
    "O141": "Pré-eclâmpsia grave",
    "O244": "Diabetes gestacional",
    "O249": "Diabetes na gravidez",
    "O470": "Falso trabalho de parto",
    "O471": "Trabalho de parto prematuro",
    "O800": "Parto cefálico espontâneo",
    "Z302": "Contraceptivo intrauterino",
    "O623": "Atonia uterina",
    "O100": "HAS pré-existente",
    "O149": "Pré-eclâmpsia NE",
    "O16":  "Hipertensão mat. NE",
    "O429": "Amniorrexe prematura NE",
    "O269": "Cond. obst. NE",
    "O479": "Trabalho de parto falso NE",
    "O034": "Aborto espontâneo completo",
}
top_cids["descricao"] = top_cids["cd_cid"].map(desc_cid).fillna(top_cids["cd_cid"])
top_cids["label"] = top_cids["cd_cid"] + " — " + top_cids["descricao"]

ax2.barh(top_cids["label"][::-1], top_cids["pct"][::-1], color="#457b9d", edgecolor="white")
ax2.set_xlabel(T["freq_xlabel"], fontsize=10)
ax2.set_title(T["top_cid_title"], fontsize=11)
ax2.tick_params(axis="y", labelsize=8)
ax2.grid(axis="x", alpha=0.3)

plt.suptitle(T["comorb_suptitle"], fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / "03_frequencia_comorbidades.png", dpi=150, bbox_inches="tight")
plt.close()
print("Gerado: 03_frequencia_comorbidades.png")

freq_grupo.to_csv(OUT_DIR / "frequencia_grupos_clinicos.csv", index=False)
top_cids.to_csv(OUT_DIR / "top_cids_frequencia.csv", index=False)
print(freq_grupo.to_string(index=False))

### 4c. Número de diagnósticos por atendimento

In [6]:
fig4, axes4 = plt.subplots(1, 2, figsize=(13, 5))

# Distribuição de qtd_diagnosticos
ax = axes4[0]
diag_dist = df["qtd_diagnosticos"].dropna().clip(0, 10).value_counts().sort_index()
ax.bar(diag_dist.index, diag_dist.values, color="#2a9d8f", edgecolor="white")
ax.set_xlabel(T["diag_xlabel"], fontsize=11)
ax.set_ylabel(T["diag_ylabel"], fontsize=11)
ax.set_title(T["diag_title"], fontsize=11)
ax.grid(axis="y", alpha=0.3)

# qtd_problemas crônicos
ax2 = axes4[1]
prob_dist = df["qtd_problemas"].dropna().clip(0, 8).value_counts().sort_index()
ax2.bar(prob_dist.index, prob_dist.values, color="#e76f51", edgecolor="white")
ax2.set_xlabel(T["prob_xlabel"], fontsize=11)
ax2.set_ylabel(T["prob_ylabel"], fontsize=11)
ax2.set_title(T["prob_title"], fontsize=11)
ax2.grid(axis="y", alpha=0.3)

plt.suptitle(T["carga_suptitle"], fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / "04_carga_diagnosticos.png", dpi=150, bbox_inches="tight")
plt.close()
print("Gerado: 04_carga_diagnosticos.png")

print("\n=== Diagnósticos por atendimento ===")
print(df["qtd_diagnosticos"].describe(percentiles=[.5,.75,.9,.95]))
print("\n=== Problemas crônicos ===")
print(df["qtd_problemas"].describe(percentiles=[.5,.75,.9,.95]))

### 4d. Comorbidades e associação com retorno em 30d

In [7]:
# Taxa de retorno 30d por grupo clínico — com IC 95% binomial (Wilson)
from scipy.stats import norm as _norm
z95 = _norm.ppf(0.975)

rows_assoc = []
for grupo in df["grupo"].unique():
    sub = df[df["grupo"] == grupo]
    n   = len(sub)
    k   = sub["y_30d"].sum()
    p   = k / n
    # Wilson interval
    denom = 1 + z95**2 / n
    centro = (p + z95**2 / (2*n)) / denom
    delta  = z95 * np.sqrt(p*(1-p)/n + z95**2/(4*n**2)) / denom
    rows_assoc.append({
        "grupo": grupo, "n": n, "retornos_30d": int(k),
        "taxa_30d": round(p*100, 2),
        "ic95_lo": round((centro - delta)*100, 2),
        "ic95_hi": round((centro + delta)*100, 2),
    })

df_assoc = pd.DataFrame(rows_assoc).sort_values("taxa_30d", ascending=False)
df_assoc.to_csv(OUT_DIR / "retorno_30d_por_grupo_ic95.csv", index=False)

fig5, ax5 = plt.subplots(figsize=(10, 5))
x5 = np.arange(len(df_assoc))
cores_assoc = [CORES_GRUPO.get(g, "#adb5bd") for g in df_assoc["grupo"]]
ax5.bar(x5, df_assoc["taxa_30d"], color=cores_assoc, edgecolor="white", zorder=2)
ax5.errorbar(x5, df_assoc["taxa_30d"],
             yerr=[df_assoc["taxa_30d"] - df_assoc["ic95_lo"],
                   df_assoc["ic95_hi"] - df_assoc["taxa_30d"]],
             fmt="none", color="black", capsize=4, lw=1.5, zorder=3)
# Linha de referência: taxa geral
taxa_geral_30d = df["y_30d"].mean() * 100
ax5.axhline(taxa_geral_30d, color="black", ls="--", lw=1.2,
            label=T["assoc_media_lbl"].format(pct=taxa_geral_30d))
ax5.set_xticks(x5)
ax5.set_xticklabels(df_assoc["grupo"], rotation=30, ha="right", fontsize=10)
ax5.set_ylabel(T["assoc_ylabel"], fontsize=11)
ax5.set_title(T["assoc_title"], fontsize=12)
ax5.legend(fontsize=10); ax5.grid(axis="y", alpha=0.3)

# Anota n por barra
for i, row in enumerate(df_assoc.itertuples()):
    ax5.text(i, row.ic95_hi + 0.5, f"n={row.n:,}", ha="center", fontsize=8, color="gray")

plt.tight_layout()
plt.savefig(PLOT_DIR / "05_retorno_por_grupo_ic95.png", dpi=150, bbox_inches="tight")
plt.close()
print("Gerado: 05_retorno_por_grupo_ic95.png")
print(df_assoc.to_string(index=False))

### 4e. Co-ocorrência de grupos clínicos (diagnósticos secundários)

Para pacientes com múltiplos diagnósticos, quais grupos aparecem juntos?

In [8]:
# Explode cids_diagnostico para análise de co-ocorrência
con2 = duckdb.connect(str(DUCKDB), read_only=True)
df_multi = con2.execute("""
    SELECT cd_atendimento, cd_paciente,
           sn_retorno_30d,
           cids_diagnostico,
           cid_secundario_laudo
    FROM gestante_enriquecido
    WHERE cids_diagnostico IS NOT NULL
       OR cid_secundario_laudo IS NOT NULL
""").df()
con2.close()

# Monta lista de todos os CIDs por atendimento (principal + diagnósticos + laudo)
def extrai_cids(row):
    cids = set()
    for campo in ["cids_diagnostico", "cid_secundario_laudo"]:
        val = row.get(campo)
        if val and str(val).strip():
            for c in str(val).replace(";",",").split(","):
                c = c.strip()
                if c:
                    cids.add(c[:3])  # Prefixo de 3 chars
    return list(cids)

df_multi["cids_lista"] = df_multi.apply(extrai_cids, axis=1)
df_multi_exp = df_multi.explode("cids_lista").dropna(subset=["cids_lista"])
df_multi_exp = df_multi_exp[df_multi_exp["cids_lista"].str.len() >= 3]

# Top 15 CIDs secundários mais frequentes
top_sec = (df_multi_exp["cids_lista"]
           .value_counts()
           .head(15)
           .reset_index())
top_sec.columns = ["cid_prefix", "n"]
top_sec["pct"] = top_sec["n"] / n_total * 100
top_sec["grupo"] = top_sec["cid_prefix"].apply(lambda c: mapeia_grupo_cid(c))

fig6, ax6 = plt.subplots(figsize=(10, 5))
cores6 = [CORES_GRUPO.get(mapeia_grupo_cid(c), "#adb5bd") for c in top_sec["cid_prefix"]]
ax6.bar(top_sec["cid_prefix"], top_sec["pct"], color=cores6, edgecolor="white")
ax6.set_xlabel(T["sec_xlabel"], fontsize=11)
ax6.set_ylabel(T["sec_ylabel"], fontsize=11)
ax6.set_title(T["sec_title"], fontsize=11)
ax6.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(PLOT_DIR / "06_cids_secundarios.png", dpi=150, bbox_inches="tight")
plt.close()
print("Gerado: 06_cids_secundarios.png")
top_sec.to_csv(OUT_DIR / "cids_secundarios_frequencia.csv", index=False)
print(top_sec.to_string(index=False))

## 5. Resumo executivo

## 6. Sazonalidade das Patologias por Mês — Sobrepondo Anos

**Chart A:** Pequenos múltiplos — um painel por ano, barras empilhadas por grupo clínico em cada mês.
Permite comparar se o padrão sazonal se mantém estável de 2017 a 2023.

**Chart B:** Um painel por grupo clínico, com uma linha por ano sobreposta.
Mostra a tendência mensal de cada patologia com anos sobrepostos.

In [9]:
ANOS     = sorted(df["ano"].dropna().astype(int).unique())
GRUPOS_ORDEM = [g for g in [
    "Hipertensivos","Diabetes Gest.","Parto Prematuro",
    "Hemorragia","Comorbidades","Pré-natal","Outros Obstétricos","Infecção"
] if g in df["grupo"].unique()]

# Monta tabela base: ano × mês × grupo → n e proporção dentro do mês-ano
df_saz = (df.groupby(["ano","mes","grupo"])
           .size()
           .reset_index(name="n"))
# Total por ano-mês (denominador para proporção)
total_mes_ano = df.groupby(["ano","mes"]).size().reset_index(name="total")
df_saz = df_saz.merge(total_mes_ano, on=["ano","mes"])
df_saz["prop"] = df_saz["n"] / df_saz["total"]

# Pivô: linhas = (ano, mes), colunas = grupo
df_pivot = (df_saz.pivot_table(index=["ano","mes"],
                                columns="grupo",
                                values="prop",
                                fill_value=0)
                  .reset_index())
# Garante colunas na ordem definida
for g in GRUPOS_ORDEM:
    if g not in df_pivot.columns:
        df_pivot[g] = 0.0
df_pivot = df_pivot[["ano","mes"] + GRUPOS_ORDEM]

# ── Chart A: Pequenos múltiplos (1 painel/ano) ───────────────────────────────
NCOLS_A = 4
NROWS_A = (len(ANOS) + NCOLS_A - 1) // NCOLS_A
fig_a, axes_a = plt.subplots(NROWS_A, NCOLS_A,
                              figsize=(5.5 * NCOLS_A, 4 * NROWS_A),
                              sharey=True)
axes_a_flat = axes_a.flatten()

for idx, ano in enumerate(ANOS):
    ax = axes_a_flat[idx]
    sub = df_pivot[df_pivot["ano"] == ano].sort_values("mes")
    bottom = np.zeros(len(sub))
    meses_x = np.arange(1, 13)

    # Garante que todos os 12 meses estejam presentes (fill 0 se faltar)
    sub_full = pd.DataFrame({"mes": meses_x})
    sub_full = sub_full.merge(sub, on="mes", how="left").fillna(0)

    bottom = np.zeros(12)
    for grupo in GRUPOS_ORDEM:
        vals = sub_full[grupo].values
        cor  = CORES_GRUPO.get(grupo, "#adb5bd")
        ax.bar(meses_x, vals * 100, bottom=bottom * 100,
               color=cor, edgecolor="white", lw=0.3,
               label=grupo, width=0.85)
        bottom += vals

    total_ano = df[df["ano"] == ano].shape[0]
    ax.set_title(f"{ano}  (n={total_ano:,})", fontsize=10, fontweight="bold")
    ax.set_xticks(meses_x)
    ax.set_xticklabels(T["meses"], fontsize=7, rotation=45)
    ax.set_ylabel(T["saz_a_ylabel"] if idx % NCOLS_A == 0 else "", fontsize=9)
    ax.set_ylim(0, 105)
    ax.grid(axis="y", alpha=0.25)

# Oculta painéis extras
for idx in range(len(ANOS), len(axes_a_flat)):
    axes_a_flat[idx].set_visible(False)

# Legenda única no último painel visível ou abaixo
handles = [plt.Rectangle((0,0),1,1, color=CORES_GRUPO.get(g,"#adb5bd")) for g in GRUPOS_ORDEM]
fig_a.legend(handles, GRUPOS_ORDEM, loc="lower center", ncol=4,
             fontsize=9, bbox_to_anchor=(0.5, -0.04), frameon=True)

plt.suptitle(T["saz_a_suptitle"], fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / "07_sazonalidade_empilhada_por_ano.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("Gerado: 07_sazonalidade_empilhada_por_ano.png")

# ── Chart B: Anos sobrepostos por grupo clínico ───────────────────────────────
GRUPOS_PLOT_B = [g for g in GRUPOS_ORDEM if g != "Infecção"]  # Infecção: n muito pequeno
NCOLS_B = 3
NROWS_B = (len(GRUPOS_PLOT_B) + NCOLS_B - 1) // NCOLS_B

CORES_ANOS = {
    2017: "#1d3557", 2018: "#457b9d", 2019: "#a8dadc",
    2020: "#f4a261", 2021: "#e76f51", 2022: "#2a9d8f", 2023: "#606c38",
}

fig_b, axes_b = plt.subplots(NROWS_B, NCOLS_B,
                              figsize=(5.5 * NCOLS_B, 4 * NROWS_B),
                              sharey=False)
axes_b_flat = axes_b.flatten()

for idx, grupo in enumerate(GRUPOS_PLOT_B):
    ax = axes_b_flat[idx]
    for ano in ANOS:
        sub = df_saz[(df_saz["grupo"] == grupo) & (df_saz["ano"] == ano)].sort_values("mes")
        if sub.empty:
            continue
        # Garante 12 pontos
        sub_full = pd.DataFrame({"mes": range(1,13)}).merge(sub, on="mes", how="left")
        sub_full["prop"] = sub_full["prop"].fillna(0)
        lw  = 2.5 if ano == 2023 else 1.4
        ls  = "--" if ano == 2023 else "-"
        alpha = 1.0 if ano >= 2021 else 0.55
        ax.plot(range(1,13), sub_full["prop"] * 100,
                color=CORES_ANOS.get(ano, "#888"),
                lw=lw, ls=ls, alpha=alpha,
                marker="o", markersize=3,
                label=str(ano))

    ax.set_title(grupo, fontsize=10, fontweight="bold",
                 color=CORES_GRUPO.get(grupo, "#333"))
    ax.set_xticks(range(1,13))
    ax.set_xticklabels(T["meses"], fontsize=8, rotation=45)
    ax.set_ylabel(T["saz_b_ylabel"] if idx % NCOLS_B == 0 else "", fontsize=9)
    ax.grid(alpha=0.25)
    ax.legend(fontsize=7, ncol=2, loc="upper right")

for idx in range(len(GRUPOS_PLOT_B), len(axes_b_flat)):
    axes_b_flat[idx].set_visible(False)

plt.suptitle(T["saz_b_suptitle"], fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / "08_sazonalidade_anos_sobrepostos.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("Gerado: 08_sazonalidade_anos_sobrepostos.png")

# Tabela: proporção média por mês (poolado) por grupo
df_saz_media = (df_saz.groupby(["mes","grupo"])
                .agg(prop_media=("prop","mean"),
                     prop_std=("prop","std"))
                .reset_index())
df_saz_media.to_csv(OUT_DIR / "sazonalidade_prop_media_por_mes.csv", index=False)
print("Gerado: sazonalidade_prop_media_por_mes.csv")

## 7. Resumo executivo



In [10]:
print("\n" + "="*65)
print("RESUMO — 01.1 Análise Temporal + Comorbidades")
print("="*65)

retorno_30  = df["y_30d"].mean()
retorno_90  = df["y_90d"].mean()
mediana_dias = df["dias_ate_retorno"].median()

print(f"Coorte total analisada    : {n_total:>8,} atendimentos")
print(f"Taxa retorno 30d          : {retorno_30:>8.1%}")
print(f"Taxa retorno 90d          : {retorno_90:>8.1%}")
print(f"Mediana dias ao retorno   : {mediana_dias:>8.0f} dias")
print(f"Grupo com maior retorno   : {df_assoc.iloc[0]['grupo']} ({df_assoc.iloc[0]['taxa_30d']}%)")
print(f"Grupo com menor retorno   : {df_assoc.iloc[-1]['grupo']} ({df_assoc.iloc[-1]['taxa_30d']}%)")
print(f"Mediana diagnósticos/atend: {df['qtd_diagnosticos'].median():>8.0f}")
print(f"% com ≥2 diagnósticos     : {(df['qtd_diagnosticos']>=2).mean():>8.1%}")
print()
print("Arquivos salvos em out/01.1_temporal_comorbidades/")
for f in sorted(PLOT_DIR.glob("*.png")): print(f"  plots/{f.name}")
for f in sorted(OUT_DIR.glob("*.csv")):  print(f"  {f.name}")